# Speech Enhancement Demo — Audio Analysis

This notebook loads **pre-computed enhanced audio** from all 4 conditioning variants,
and provides a detailed comparison via:
1. **Audio playback** (IPython.display)
2. **Mel-spectrogram** comparison
3. **Spectral difference** analysis (enhanced vs. clean)
4. **Waveform** overlay and zoom
5. **Per-sample PESQ & STOI** scores
6. **Frequency band energy** profiles

## Setup

Set `OUTPUTS_ROOT` below to point to your local copy of the outputs.
Download from: https://drive.google.com/drive/folders/1uH9JNkk6FzoSQb3anljAYGUmRdeIPw4d

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display
import soundfile as sf
import IPython.display as ipd

try:
    from pesq import pesq
    HAS_PESQ = True
except ImportError:
    HAS_PESQ = False
    print('pesq not installed - skipping PESQ computation')

try:
    from pystoi import stoi
    HAS_STOI = True
except ImportError:
    HAS_STOI = False
    print('pystoi not installed - skipping STOI computation')

plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (14, 4)

SR = 16000  # all audio is 16 kHz

## Configuration

In [ ]:
# ---- CHANGE THESE PATHS ----
# Root of the speech enhancement outputs (download from Google Drive)
OUTPUTS_ROOT = 'speech_enhancement_outputs/audio_samples'

# Sample to analyse (filename stem without .wav)
SAMPLE_STEM = 'LibriSpeech_dev-clean_6319_275224_6319-275224-0014'

# Path to the original noisy mixture (from data/mixed/)
NOISY_WAV = f'data/mixed/snr_5dB/{SAMPLE_STEM}.wav'

# Path to the original clean speech
CLEAN_FLAC = 'data/raw/clean/LibriSpeech/dev-clean/6319/275224/6319-275224-0014.flac'

# Condition types to compare
CONDITIONS = ['none', 'last_layer', 'multi_layer', 'multi_layer_time']
CONDITION_LABELS = ['No Cond.', 'Last Layer', 'Static ML', 'Time-dep. ML']
# ----------------------------

def load_enhanced(condition, stem):
    path = os.path.join(OUTPUTS_ROOT, condition, 'wavs', f'{stem}.wav')
    if os.path.exists(path):
        wav, _ = librosa.load(path, sr=SR)
        return wav
    alt = os.path.join(OUTPUTS_ROOT, f'{condition}_{stem}.wav')
    if os.path.exists(alt):
        wav, _ = librosa.load(alt, sr=SR)
        return wav
    print(f'WARNING: not found: {path}')
    return None

print('Configuration OK')

## 1. Load Audio

In [ ]:
clean_wav, _ = librosa.load(CLEAN_FLAC, sr=SR)
noisy_wav, _ = librosa.load(NOISY_WAV, sr=SR)

enhanced = {}
for ct in CONDITIONS:
    wav = load_enhanced(ct, SAMPLE_STEM)
    if wav is not None:
        enhanced[ct] = wav

print(f'Clean:  {len(clean_wav)/SR:.2f}s')
print(f'Noisy:  {len(noisy_wav)/SR:.2f}s')
for ct, wav in enhanced.items():
    print(f'{ct:20s}: {len(wav)/SR:.2f}s')

## 2. Audio Playback

Listen to each variant. Compare the noise level and speech clarity.

In [ ]:
print('Clean (reference)')
ipd.display(ipd.Audio(clean_wav, rate=SR))

print('\nNoisy (5 dB SNR input)')
ipd.display(ipd.Audio(noisy_wav, rate=SR))

for ct, label in zip(CONDITIONS, CONDITION_LABELS):
    if ct in enhanced:
        print(f'\nEnhanced - {label}')
        ipd.display(ipd.Audio(enhanced[ct], rate=SR))

## 3. Mel-Spectrogram Comparison

Side-by-side log-Mel spectrograms for all variants.

In [ ]:
def compute_mel(wav, sr=SR):
    mel = librosa.feature.melspectrogram(y=wav, sr=sr, n_fft=1024, hop_length=256, n_mels=80)
    return librosa.power_to_db(mel, ref=np.max)

panels = [('Clean', clean_wav), ('Noisy (5 dB)', noisy_wav)]
for ct, label in zip(CONDITIONS, CONDITION_LABELS):
    if ct in enhanced:
        panels.append((f'Enhanced\n{label}', enhanced[ct]))

n = len(panels)
fig, axes = plt.subplots(1, n, figsize=(4.5 * n, 3.5))
if n == 1:
    axes = [axes]

for ax, (label, wav) in zip(axes, panels):
    mel_db = compute_mel(wav)
    img = librosa.display.specshow(mel_db, sr=SR, hop_length=256, x_axis='time', y_axis='mel', ax=ax)
    ax.set_title(label, fontsize=9)
    fig.colorbar(img, ax=ax, format='%+2.0f dB')

fig.suptitle(f'Mel-Spectrogram Comparison - {SAMPLE_STEM}', fontsize=11)
plt.tight_layout()
plt.show()

## 4. Spectral Difference (Enhanced vs. Clean)

Visualise the **residual** between each enhanced variant and the clean reference.
Smaller (darker) residuals indicate better reconstruction.

In [ ]:
clean_mel = compute_mel(clean_wav)

diff_panels = []
for ct, label in zip(CONDITIONS, CONDITION_LABELS):
    if ct in enhanced:
        enh_mel = compute_mel(enhanced[ct])
        min_t = min(clean_mel.shape[1], enh_mel.shape[1])
        diff = np.abs(enh_mel[:, :min_t] - clean_mel[:, :min_t])
        diff_panels.append((label, diff))

noisy_mel = compute_mel(noisy_wav)
min_t = min(clean_mel.shape[1], noisy_mel.shape[1])
noisy_diff = np.abs(noisy_mel[:, :min_t] - clean_mel[:, :min_t])

n = 1 + len(diff_panels)
fig, axes = plt.subplots(1, n, figsize=(4.5 * n, 3.5))

img = librosa.display.specshow(noisy_diff, sr=SR, hop_length=256, x_axis='time', y_axis='mel', ax=axes[0])
axes[0].set_title('Noisy - Clean', fontsize=9)
fig.colorbar(img, ax=axes[0], format='%+2.0f dB')

for ax, (label, diff) in zip(axes[1:], diff_panels):
    img = librosa.display.specshow(diff, sr=SR, hop_length=256, x_axis='time', y_axis='mel', ax=ax)
    ax.set_title(f'{label} - Clean', fontsize=9)
    fig.colorbar(img, ax=ax, format='%+2.0f dB')

fig.suptitle('Spectral Residual (|Enhanced - Clean|)', fontsize=11)
plt.tight_layout()
plt.show()

## 5. Waveform Comparison

Time-domain waveforms stacked vertically.

In [ ]:
all_wavs = [('Clean', clean_wav), ('Noisy', noisy_wav)]
for ct, label in zip(CONDITIONS, CONDITION_LABELS):
    if ct in enhanced:
        all_wavs.append((label, enhanced[ct]))

n = len(all_wavs)
fig, axes = plt.subplots(n, 1, figsize=(14, 2.2 * n), sharex=True)

for ax, (label, wav) in zip(axes, all_wavs):
    t = np.arange(len(wav)) / SR
    ax.plot(t, wav, linewidth=0.3, color='steelblue')
    ax.set_ylabel(label, fontsize=8, rotation=0, labelpad=70, va='center')
    ax.set_xlim(0, t[-1])
    ax.set_ylim(-1, 1)
    ax.tick_params(labelsize=7)

axes[-1].set_xlabel('Time (s)')
fig.suptitle('Waveform Comparison', fontsize=11)
plt.tight_layout()
plt.show()

## 6. Zoomed-In Waveform Segment

Zoom into a 0.5s segment to see fine-grained waveform detail.

In [ ]:
start_sec = 1.0
dur_sec = 0.5
s0 = int(start_sec * SR)
s1 = int((start_sec + dur_sec) * SR)

fig, axes = plt.subplots(len(all_wavs), 1, figsize=(14, 2.0 * len(all_wavs)), sharex=True)

for ax, (label, wav) in zip(axes, all_wavs):
    seg = wav[s0:s1] if len(wav) > s1 else wav[s0:]
    t = np.arange(len(seg)) / SR + start_sec
    ax.plot(t, seg, linewidth=0.6, color='steelblue')
    ax.set_ylabel(label, fontsize=8, rotation=0, labelpad=70, va='center')
    ax.tick_params(labelsize=7)

axes[-1].set_xlabel('Time (s)')
fig.suptitle(f'Zoomed Waveform ({start_sec:.1f}-{start_sec+dur_sec:.1f}s)', fontsize=11)
plt.tight_layout()
plt.show()

## 7. Per-Sample Metrics (PESQ & STOI)

In [ ]:
print(f"{'Variant':<20s}  {'PESQ':>8s}  {'STOI':>8s}")
print(f"{'-'*20}  {'-'*8}  {'-'*8}")

min_len = min(len(clean_wav), len(noisy_wav))
noisy_pesq = pesq(SR, clean_wav[:min_len], noisy_wav[:min_len], 'wb') if HAS_PESQ else float('nan')
noisy_stoi = stoi(clean_wav[:min_len], noisy_wav[:min_len], SR, extended=False) if HAS_STOI else float('nan')
print(f"{'Noisy input':<20s}  {noisy_pesq:8.4f}  {noisy_stoi:8.4f}")

for ct, label in zip(CONDITIONS, CONDITION_LABELS):
    if ct not in enhanced:
        continue
    enh = enhanced[ct]
    ml = min(len(clean_wav), len(enh))
    p = pesq(SR, clean_wav[:ml], enh[:ml], 'wb') if HAS_PESQ else float('nan')
    s = stoi(clean_wav[:ml], enh[:ml], SR, extended=False) if HAS_STOI else float('nan')
    print(f"{label:<20s}  {p:8.4f}  {s:8.4f}")

## 8. Frequency Band Energy Comparison

Compare how well each variant preserves energy across frequency bands.

In [ ]:
def band_energy(wav, sr, n_fft=1024):
    mel = librosa.feature.melspectrogram(y=wav, sr=sr, n_fft=n_fft, hop_length=256, n_mels=80)
    return np.mean(mel, axis=1)

clean_energy = band_energy(clean_wav, SR)
noisy_energy = band_energy(noisy_wav, SR)

fig, ax = plt.subplots(figsize=(10, 4))
mel_freqs = librosa.mel_frequencies(n_mels=80, fmax=SR/2)

ax.semilogy(mel_freqs, clean_energy, label='Clean', linewidth=1.5, color='green')
ax.semilogy(mel_freqs, noisy_energy, label='Noisy', linewidth=1.0, color='gray', alpha=0.7)

colors = ['#e74c3c', '#e67e22', '#2980b9', '#8e44ad']
for (ct, label), color in zip(zip(CONDITIONS, CONDITION_LABELS), colors):
    if ct in enhanced:
        eng = band_energy(enhanced[ct], SR)
        ax.semilogy(mel_freqs, eng, label=label, linewidth=1.0, color=color)

ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Mean Energy')
ax.set_title('Frequency Band Energy Profiles')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Summary

Key observations from this sample:

1. **No conditioning** removes some noise but introduces artefacts and loses speech detail.
2. **Last-layer** conditioning improves clarity but still misses fine-grained acoustic structure.
3. **Static multi-layer** captures more spectral detail - the spectral residual is visibly smaller.
4. **Time-dependent multi-layer** produces the closest match to clean speech in both the spectrogram and waveform views.

The progression confirms our core finding: **multi-layer tokenizer features provide complementary information** that progressively improves enhancement quality.